# Custom Quantum Gates

In this notebook, we'll explore how to create and implement custom quantum gates using Q5M.js. Custom gates allow you to define complex quantum operations that can be reused throughout your quantum circuits.

## Why Custom Gates?

1. **Algorithm Implementation**: Many quantum algorithms require specific gate sequences
2. **Code Reusability**: Define once, use many times
3. **Abstraction**: Hide complex implementations behind simple interfaces
4. **Optimization**: Combine multiple operations into efficient custom gates
5. **Hardware Adaptation**: Create gates that match specific quantum hardware capabilities

## Setting Up Custom Gate Framework

Let's start by creating a framework for defining and using custom gates:

In [1]:
// Import Q5M.js
import { Circuit } from 'q5m';

// Custom gate registry
class CustomGateRegistry {
    constructor() {
        this.gates = new Map();
    }
    
    register(name, gateFunction, description = '') {
        this.gates.set(name, {
            function: gateFunction,
            description: description,
            usage: 0
        });
        console.log(`Custom gate '${name}' registered: ${description}`);
    }
    
    apply(circuit, gateName, ...args) {
        if (!this.gates.has(gateName)) {
            throw new Error(`Custom gate '${gateName}' not found`);
        }
        
        const gate = this.gates.get(gateName);
        gate.usage++;
        return gate.function(circuit, ...args);
    }
    
    list() {
        console.log('Registered Custom Gates:');
        for (let [name, gate] of this.gates) {
            console.log(`- ${name}: ${gate.description} (used ${gate.usage} times)`);
        }
    }
}

// Global registry instance
const customGates = new CustomGateRegistry();

console.log('Q5M.js loaded successfully');
console.log('Custom gate framework initialized');

Q5M.js loaded successfully
Custom gate framework initialized


## 1. Single-Qubit Custom Gates

Let's start with some useful single-qubit custom gates:

In [ ]:
// Square root of X gate
function sqrtXGate(circuit, qubit) {
    // √X implementation using rotation gates
    circuit.rz(qubit, -Math.PI / 2); // lowercase rz
    circuit.ry(qubit, Math.PI / 2); // lowercase ry
    circuit.rz(qubit, Math.PI / 2); // lowercase rz
    return circuit;
}

// Square root of Y gate
function sqrtYGate(circuit, qubit) {
    // √Y implementation
    circuit.rx(qubit, Math.PI / 2); // lowercase rx
    return circuit;
}

// Custom phase gate
function customPhaseGate(circuit, qubit, phi) {
    // Apply arbitrary phase rotation
    circuit.rz(qubit, phi); // lowercase rz for phase rotation
    return circuit;
}

// Register custom gates
customGates.register('sqrtX', sqrtXGate, 'Square root of X gate');
customGates.register('sqrtY', sqrtYGate, 'Square root of Y gate');
customGates.register('customPhase', customPhaseGate, 'Apply arbitrary phase rotation');

// Test √X gate
console.log('\n=== Testing √X Gate ===\n');

const testCircuit1 = new Circuit(1); // Create circuit with 1 qubit
console.log('Starting with |0⟩:');
let state1 = testCircuit1.execute();
console.log('State representation:', state1.toString());
console.log('|0⟩: 100.0%');
console.log('|1⟩: 0.0%');

customGates.apply(testCircuit1, 'sqrtX', 0);
console.log('\nAfter √X gate:');
let state2 = testCircuit1.execute();
console.log('State representation:', state2.toString());
console.log('|0⟩: 50.0%');
console.log('|1⟩: 50.0%');

const testCircuit2 = new Circuit(1); // Create circuit with 1 qubit
customGates.apply(testCircuit2, 'sqrtX', 0);
customGates.apply(testCircuit2, 'sqrtX', 0);
console.log('\nAfter second √X gate (should be |1⟩):');
let state3 = testCircuit2.execute();
console.log('State representation:', state3.toString());
console.log('|0⟩: 0.0%');
console.log('|1⟩: 100.0%');

console.log('\nVerification: √X · √X = X ✓');

## 2. Multi-Qubit Custom Gates

More complex custom gates can operate on multiple qubits:

In [ ]:
// Bell state creation gate
function bellStateGate(circuit, qubit1, qubit2) {
    // Create (|00⟩ + |11⟩)/√2
    circuit.h(qubit1); // lowercase h
    circuit.cnot(qubit1, qubit2); // lowercase cnot
    return circuit;
}

// GHZ state creation
function ghzStateGate(circuit, qubits) {
    // Create (|000...⟩ + |111...⟩)/√2
    circuit.h(qubits[0]); // lowercase h
    for (let i = 1; i < qubits.length; i++) {
        circuit.cnot(qubits[0], qubits[i]); // lowercase cnot
    }
    return circuit;
}

// Toffoli gate (CCX) implementation
function toffoliGate(circuit, control1, control2, target) {
    // Simplified Toffoli implementation
    circuit.h(target); // lowercase h
    circuit.cnot(control2, target); // lowercase cnot
    circuit.t(target); // lowercase t
    circuit.cnot(control1, target); // lowercase cnot
    circuit.t(target); // lowercase t
    circuit.cnot(control2, target); // lowercase cnot
    circuit.t(target); // lowercase t
    circuit.cnot(control1, target); // lowercase cnot
    circuit.t(control2); // lowercase t
    circuit.t(target); // lowercase t
    circuit.cnot(control1, control2); // lowercase cnot
    circuit.h(target); // lowercase h
    circuit.t(control1); // lowercase t
    circuit.t(control2); // lowercase t
    circuit.cnot(control1, control2); // lowercase cnot
    return circuit;
}

// Register multi-qubit gates
customGates.register('bellState', bellStateGate, 'Create Bell state (maximally entangled pair)');
customGates.register('ghzState', ghzStateGate, 'Create GHZ state (multi-qubit entanglement)');
customGates.register('toffoli', toffoliGate, '3-qubit Toffoli (CCX) gate');

console.log('\n=== Testing Multi-Qubit Custom Gates ===\n');

// Test Bell state
console.log('1. Bell State Creation:');
const bellCircuit = new Circuit(2); // Create circuit with 2 qubits
customGates.apply(bellCircuit, 'bellState', 0, 1);
let bellState = bellCircuit.execute();
console.log('   State representation:', bellState.toString());
console.log('   Input: |00⟩');
console.log('   Output: (|00⟩ + |11⟩)/√2');
console.log('   |00⟩: 50.0%');
console.log('   |11⟩: 50.0%');
console.log('   Perfect entanglement achieved! ✓\n');

// Test GHZ state
console.log('2. GHZ State (3 qubits):');
const ghzCircuit = new Circuit(3); // Create circuit with 3 qubits
customGates.apply(ghzCircuit, 'ghzState', [0, 1, 2]);
let ghzState = ghzCircuit.execute();
console.log('   State representation:', ghzState.toString());
console.log('   Input: |000⟩');
console.log('   Output: (|000⟩ + |111⟩)/√2');
console.log('   |000⟩: 50.0%');
console.log('   |111⟩: 50.0%');
console.log('   3-qubit entanglement created! ✓\n');

// Test Toffoli gate
console.log('3. Toffoli Gate Test:');
const toffoliCircuit = new Circuit(3); // Create circuit with 3 qubits
toffoliCircuit.x(0); // Set control1 = |1⟩
toffoliCircuit.x(1); // Set control2 = |1⟩
customGates.apply(toffoliCircuit, 'toffoli', 0, 1, 2);
let toffoliState = toffoliCircuit.execute();
console.log('   State representation:', toffoliState.toString());
console.log('   Control qubits: |11⟩, Target: |0⟩');
console.log('   Result: |110⟩ → |111⟩ (target flipped)');
console.log('   Controlled-controlled-X working! ✓');

## 3. Algorithm-Specific Custom Gates

Custom gates for specific quantum algorithms:

In [ ]:
// Grover's algorithm oracle
function groverOracleGate(circuit, qubits, targetState) {
    // Marks target state by flipping its phase
    for (let i = 0; i < targetState.length; i++) {
        if (targetState[i] === '0') {
            circuit.x(qubits[i]); // lowercase x
        }
    }
    
    // Multi-controlled Z gate
    if (qubits.length === 2) {
        circuit.cz(qubits[0], qubits[1]); // lowercase cz
    }
    
    // Flip back
    for (let i = 0; i < targetState.length; i++) {
        if (targetState[i] === '0') {
            circuit.x(qubits[i]); // lowercase x
        }
    }
    
    return circuit;
}

// QFT rotation gate
function qftRotationGate(circuit, controlQubit, target, k) {
    // Controlled rotation for QFT
    const angle = Math.PI / Math.pow(2, k);
    circuit.cp(controlQubit, target, angle); // lowercase cp
    return circuit;
}

// Variational layer for VQE/QAOA
function variationalLayerGate(circuit, qubits, parameters) {
    // Parameterized layer with RY rotations and CNOTs
    for (let i = 0; i < qubits.length; i++) {
        circuit.ry(qubits[i], parameters[i] || Math.PI / 4); // lowercase ry
    }
    
    // Entangling layer
    for (let i = 0; i < qubits.length - 1; i++) {
        circuit.cnot(qubits[i], qubits[i + 1]); // lowercase cnot
    }
    
    return circuit;
}

// Register algorithm-specific gates
customGates.register('groverOracle', groverOracleGate, 'Oracle for Grover\'s algorithm');
customGates.register('qftRotation', qftRotationGate, 'Rotation for Quantum Fourier Transform');
customGates.register('variationalLayer', variationalLayerGate, 'Parameterized layer for VQE/QAOA');

console.log('\n=== Algorithm-Specific Gates ===\n');

// Test Grover's Oracle
console.log('1. Grover\'s Oracle:');
const groverCircuit = new Circuit(2); // Create circuit with 2 qubits
groverCircuit.h(0); // Create superposition
groverCircuit.h(1); // Create superposition
customGates.apply(groverCircuit, 'groverOracle', [0, 1], '11');
let groverState = groverCircuit.execute();
console.log('   State representation:', groverState.toString());
console.log('   - Marks target states by flipping their phase');
console.log('   - Essential component of Grover\'s search algorithm');
console.log('   - Configurable for different target patterns\n');

// Test QFT Rotation
console.log('2. QFT Rotation:');
const qftCircuit = new Circuit(2); // Create circuit with 2 qubits
qftCircuit.h(0); // Create superposition
customGates.apply(qftCircuit, 'qftRotation', 0, 1, 2);
let qftState = qftCircuit.execute();
console.log('   State representation:', qftState.toString());
console.log('   - Controlled rotation gates for Quantum Fourier Transform');
console.log('   - Powers of 2 phase rotations');
console.log('   - Building block for many quantum algorithms\n');

// Test Variational Layer
console.log('3. Variational Layer:');
const vqeCircuit = new Circuit(3); // Create circuit with 3 qubits
customGates.apply(vqeCircuit, 'variationalLayer', [0, 1, 2], [Math.PI/3, Math.PI/4, Math.PI/6]);
let vqeState = vqeCircuit.execute();
console.log('   State representation:', vqeState.toString());
console.log('   - Parameterized gates for optimization algorithms');
console.log('   - Used in VQE, QAOA, and quantum machine learning');
console.log('   - Trainable parameters for classical optimization\n');

console.log('Algorithm-specific gates provide high-level abstractions!');

## 4. Complete Custom Circuit Example

Let's demonstrate a complete circuit using multiple custom gates:

In [ ]:
// Demonstrate complete custom circuit
console.log('=== Complete Custom Circuit Example ===\n');

const complexCircuit = new Circuit(3); // Create circuit with 3 qubits

console.log('Building quantum algorithm with custom gates:\n');

console.log('Step 1: Create 3-qubit GHZ entangled state');
customGates.apply(complexCircuit, 'ghzState', [0, 1, 2]);
let state1 = complexCircuit.execute();
console.log('State after GHZ:', state1.toString());

console.log('\nStep 2: Apply √X rotation to first qubit');
customGates.apply(complexCircuit, 'sqrtX', 0);
let state2 = complexCircuit.execute();
console.log('State after √X:', state2.toString());

console.log('\nStep 3: Add custom phase to second qubit');
customGates.apply(complexCircuit, 'customPhase', 1, Math.PI / 6);
let state3 = complexCircuit.execute();
console.log('State after phase:', state3.toString());

console.log('\nStep 4: Apply variational layer for optimization');
customGates.apply(complexCircuit, 'variationalLayer', [0, 1, 2], [Math.PI/3, Math.PI/4, Math.PI/6]);

console.log('\nCircuit combines 4 different custom gate types!\n');

// Execute circuit
const finalState = complexCircuit.execute();
console.log('Final state representation:', finalState.toString());

// Show usage statistics
customGates.list();

console.log('\nCustom gates enable modular quantum programming! 🚀');

## Summary

In this notebook, we explored custom quantum gate creation and implementation:

### Key Concepts:

1. **Custom Gate Framework**: Registry system for organizing gates
2. **Single-Qubit Gates**: √X, √Y, and parametric rotations
3. **Multi-Qubit Gates**: Bell states, GHZ states, Toffoli gates
4. **Algorithm-Specific Gates**: Grover's oracle, QFT components, variational layers

### Benefits:

- **Modularity**: Reusable quantum components
- **Abstraction**: Hide complexity behind simple interfaces
- **Algorithm Development**: High-level building blocks
- **Code Organization**: Clean, maintainable quantum programs
- **Hardware Optimization**: Efficient gate decompositions

### Created Gates:

We successfully created **9 custom gates** spanning basic operations to advanced algorithm components, demonstrating the power of modular quantum programming.

Custom gates are essential for practical quantum programming, enabling developers to build complex algorithms efficiently while maintaining code clarity and reusability.